In [ ]:
# this is the file i created to embedd the titles and get the faiss index files in google colab t4 gpu
import pandas as pd
import numpy as np
import re
from html import unescape

collection_anime = pd.read_csv("/content/anime.csv")
collection_manga = pd.read_csv("/content/manga.csv")

In [ ]:
import re
import pandas as pd
from html import unescape

# regex to clean the description from unnecessary data like the awards won by the title and some other notes this is done so that the embeddings are not corrupted by unnecessary data and only the description is embedded
def clean_description_for_embedding(raw_text):
    if pd.isna(raw_text) or raw_text == "":
        return "", ""

    # 1. Decode HTML entities
    text = unescape(str(raw_text))

    # 2. Extract Notes (Robust regex matching newlines)
    # We capture this BEFORE removing it so we can return it separately
    notes_pattern = r"<i[^>]*>\s*Notes?[\s\S]*?</i>"
    notes_match = re.search(notes_pattern, text, flags=re.IGNORECASE)

    # Clean up the extracted notes string (remove tags from it if you want purely text)
    notes = notes_match.group(0) if notes_match else ""

    # 3. Remove Notes from the main text
    text = re.sub(notes_pattern, "", text, flags=re.IGNORECASE)

    # 4. Remove "(Source: ...)"
    text = re.sub(r"\(\s*Source\s*:[\s\S]*?\)", "", text, flags=re.IGNORECASE)

    # 5. Remove ALL remaining HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # 6. Normalize Whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Return TWO values to match your pandas code#
    return text, notes

In [ ]:
collection_anime[["description_clean", "notes"]] = (
    collection_anime["description"]
    .apply(lambda x: pd.Series(clean_description_for_embedding(x)))
)


In [ ]:
collection_manga[["description_clean", "notes"]] = (
    collection_manga["description"]
    .apply(lambda x: pd.Series(clean_description_for_embedding(x)))
)


In [ ]:
collection_anime.to_csv("/content/anime_clean.csv", index=False)
collection_manga.to_csv("/content/manga_clean.csv", index=False)


In [ ]:
collection_manga["description_clean"][1]

"His name is Guts, the Black Swordsman, a feared warrior spoken of only in whispers. Bearer of a gigantic sword, an iron hand, and the scars of countless battles and tortures, his flesh is also indelibly marked with The Brand, an unholy symbol that draws the forces of darkness to him and dooms him as their sacrifice. But Guts won't take his fate lying down; he'll cut a crimson swath of carnage through the ranks of the damned—and anyone else foolish enough to oppose him! Accompanied by Puck the Elf, more an annoyance than a companion, Guts relentlessly follows a dark, bloodstained path that leads only to death...or vengeance."

In [ ]:
# this is to convert the tags and genres from  strings to lists, however this is not perfect since the tags were stored like this '[tag]' which is later revised in the clean_db.py script
def split_to_list(value):
    if pd.isna(value):
        return []
    return [item.strip() for item in value.split(",") if item.strip()]


In [ ]:
collection_anime["tags"] = collection_anime["tags"].apply(split_to_list)
collection_anime["genres"] = collection_anime["genres"].apply(split_to_list)

In [ ]:
collection_manga["tags"] = collection_manga["tags"].apply(split_to_list)
collection_manga["genres"] = collection_manga["genres"].apply(split_to_list)

In [ ]:
# this is done just in case i want to expand or reduce the tag / genre selection later
unique_anime_tags = set()
for atag_list in collection_anime['tags']:
  for tag in atag_list:
    unique_anime_tags.add(tag.strip())

unique_anime_tags = sorted(list(unique_anime_tags))
print(unique_anime_tags)
print(len(unique_anime_tags))

with open("unique_anime_tags.txt", "w") as f:
  for tag in unique_anime_tags:
    f.write(f"{tag}\n")

['4-koma', 'Achromatic', 'Achronological Order', 'Acrobatics', 'Acting', 'Adoption', 'Advertisement', 'Afterlife', 'Age Gap', 'Age Regression', 'Agender', 'Agriculture', 'Ahegao', 'Airsoft', 'Alchemy', 'Aliens', 'Alternate Universe', 'American Football', 'Amnesia', 'Amputation', 'Anachronism', 'Anal Sex', 'Ancient China', 'Angels', 'Animals', 'Anthology', 'Anthropomorphism', 'Anti-Hero', 'Archery', 'Armpits', 'Aromantic', 'Arranged Marriage', 'Artificial Intelligence', 'Asexual', 'Ashikoki', 'Asphyxiation', 'Assassins', 'Astronomy', 'Athletics', 'Augmented Reality', 'Autobiographical', 'Aviation', 'Badminton', 'Ballet', 'Band', 'Bar', 'Baseball', 'Basketball', 'Battle Royale', 'Biographical', 'Bisexual', 'Blackmail', 'Board Game', 'Boarding School', 'Body Horror', 'Body Image', 'Body Swapping', 'Bondage', 'Boobjob', 'Bowling', 'Boxing', "Boys' Love", 'Bullying', 'Butler', 'CGI', 'Calligraphy', 'Camping', 'Cannibalism', 'Card Battle', 'Cars', 'Centaur', 'Cervix Penetration', 'Cheating',

In [ ]:
unique_manga_tags = set()
for mtag_list in collection_manga['tags']:
  for tag in mtag_list:
    unique_manga_tags.add(tag.strip())

unique_manga_tags = sorted(list(unique_manga_tags))
print(unique_manga_tags)
print(len(unique_manga_tags))

with open("unique_manga_tags.txt", "w") as f:
  for tag in unique_manga_tags:
    f.write(f"{tag}\n")

['4-koma', 'Achromatic', 'Achronological Order', 'Acrobatics', 'Acting', 'Adoption', 'Advertisement', 'Afterlife', 'Age Gap', 'Age Regression', 'Agender', 'Agriculture', 'Ahegao', 'Airsoft', 'Alchemy', 'Aliens', 'Alternate Universe', 'American Football', 'Amnesia', 'Amputation', 'Anachronism', 'Anal Sex', 'Ancient China', 'Angels', 'Animals', 'Anthology', 'Anthropomorphism', 'Anti-Hero', 'Archery', 'Armpits', 'Aromantic', 'Arranged Marriage', 'Artificial Intelligence', 'Asexual', 'Ashikoki', 'Asphyxiation', 'Assassins', 'Astronomy', 'Athletics', 'Augmented Reality', 'Autobiographical', 'Aviation', 'Badminton', 'Ballet', 'Band', 'Bar', 'Baseball', 'Basketball', 'Battle Royale', 'Biographical', 'Bisexual', 'Blackmail', 'Board Game', 'Boarding School', 'Body Horror', 'Body Image', 'Body Swapping', 'Bondage', 'Boobjob', 'Bowling', 'Boxing', "Boys' Love", 'Bullying', 'Butler', 'CGI', 'Calligraphy', 'Camping', 'Cannibalism', 'Card Battle', 'Cars', 'Centaur', 'Cervix Penetration', 'Cheating',

In [ ]:
unique_anime_genres = set()
for atag_list in collection_anime['genres']:
  for tag in atag_list:
    unique_anime_genres.add(tag.strip())

unique_anime_genres = sorted(list(unique_anime_genres))
print(unique_anime_genres)
print(len(unique_anime_genres))

with open("unique_anime_genres.txt", "w") as f:
  for genre in unique_anime_genres:
    f.write(f"{genre}\n")

['Action', 'Adventure', 'Comedy', 'Drama', 'Ecchi', 'Fantasy', 'Hentai', 'Horror', 'Mahou Shoujo', 'Mecha', 'Music', 'Mystery', 'Psychological', 'Romance', 'Sci-Fi', 'Slice of Life', 'Sports', 'Supernatural', 'Thriller']
19


In [ ]:
unique_manga_genres = set()
for mtag_list in collection_manga['genres']:
  for tag in mtag_list:
    unique_manga_genres.add(tag.strip())

unique_manga_genres = sorted(list(unique_manga_genres))
print(unique_manga_genres)
print(len(unique_manga_genres))

with open("unique_manga_genres.txt", "w") as f:
  for genre in unique_manga_genres:
    f.write(f"{genre}\n")

['Action', 'Adventure', 'Comedy', 'Drama', 'Ecchi', 'Fantasy', 'Hentai', 'Horror', 'Mahou Shoujo', 'Mecha', 'Music', 'Mystery', 'Psychological', 'Romance', 'Sci-Fi', 'Slice of Life', 'Sports', 'Supernatural', 'Thriller']
19


In [ ]:
collection_anime["combined_text"] = collection_anime.apply(
    lambda row: (
        f"Title: {row['title']}. "
        f"Genres: {', '.join(row['genres'])}. "
        f"Tags: {', '.join(row['tags'])}. "
        f"Description: {row['description_clean']}."
    ),
    axis=1
)

In [ ]:
collection_manga["combined_text"] = collection_manga.apply(
    lambda row: (
        f"Title: {row['title']}. "
        f"Genres: {', '.join(row['genres'])}. "
        f"Tags: {', '.join(row['tags'])}. "
        f"Description: {row['description_clean']}."
    ),
    axis=1
)

In [ ]:
collection_anime['combined_text'][0]

'Title: Cowboy Bebop. Genres: Action, Adventure, Drama, Sci-Fi. Tags: Space, Crime, Episodic, Ensemble Cast, Primarily Adult Cast, Tragedy, Travel, Noir, Anti-Hero, Philosophy, Guns, Cyberpunk, Male Protagonist, Found Family, Terrorism, Female Protagonist, Cowboys, Martial Arts, Cyborg, Tomboy, Amnesia, Gambling, Heterosexual, Yakuza, Drugs, Police, Nudity, Tanned Skin, Cult, Circus, CGI, Work. Description: Enter a world in the distant future, where Bounty Hunters roam the solar system. Spike and Jet, bounty hunting partners, set out on journeys in an ever struggling effort to win bounty rewards to survive. While traveling, they meet up with other very interesting people. Could Faye, the beautiful and ridiculously poor gambler, Edward, the computer genius, and Ein, the engineered dog be a good addition to the group?.'

In [ ]:
!pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 107.7 MB/s eta 0:00:00


In [ ]:
# create the embeddings for anime titles
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device = 'cuda')

embedding_anime = embedding_model.encode(
    collection_anime['combined_text'].tolist(),
    show_progress_bar=True
)

embeddings_anime = np.array(embedding_anime)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/682 [00:00<?, ?it/s]

In [ ]:
# same for manga titles
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device = 'cuda')

embedding_manga = embedding_model.encode(
    collection_manga['combined_text'].tolist(),
    show_progress_bar=True
)

embeddings_manga = np.array(embedding_manga)

Batches:   0%|          | 0/4016 [00:00<?, ?it/s]

In [ ]:
embeddings_manga.shape
embeddings_anime.shape

(21812, 384)

In [ ]:
pip install faiss-cpu

In [ ]:
# this is to store all the embeddings created in FAISS vector store
import numpy as np
import faiss

# Check dtype and cast to float32
embeddings_anime = embeddings_anime.astype('float32', copy=False)

# calculate the magnitude (norm) for each vector separately
norms = np.linalg.norm(embeddings_anime, axis=1, keepdims=True)
norms[norms == 0] = 1e-12
embeddings_anime = embeddings_anime / norms  # now each row is unit length

# Create FAISS index (Flat L2 is fine for unit vectors; alternatively use IndexFlatIP)
d = embeddings_anime.shape[1] #384 from what we got before
index = faiss.IndexFlatL2(d)   # L2 on unit vectors ~ cosine

# Add embeddings to index
index.add(embeddings_anime)   # emb shape (N, d), dtype float32

print("Vectors in index:", index.ntotal)

# Save index to disk and save mapping 
faiss.write_index(index, "anime_index.faiss")
# also save the metadata table (collections) to CSV
collection_anime.to_csv("cleaned_anime_collections_with_combined_text.csv", index=False)
# this is optional 
np.save("embeddings.npy", embeddings_anime)

Vectors in index: 21812


In [ ]:
# store all the embeddings created in FAISS vector store
import numpy as np
import faiss

# Check dtype and cast to float32
embeddings_manga = embeddings_manga.astype('float32', copy=False)

# calculate the magnitude (norm) for each vector separately
norms = np.linalg.norm(embeddings_manga, axis=1, keepdims=True)
norms[norms == 0] = 1e-12
embeddings_manga = embeddings_manga / norms  # now each row is unit length

# Create FAISS index (Flat L2 is fine for unit vectors; alternatively use IndexFlatIP)
d = embeddings_manga.shape[1] #384 from what we got before
index = faiss.IndexFlatL2(d)   # L2 on unit vectors ~ cosine

# Add embeddings to index
index.add(embeddings_manga)   # emb shape (N, d), dtype float32

print("Vectors in index:", index.ntotal)

# Save index to disk and save mapping (optional but recommended)
faiss.write_index(index, "manga_index.faiss")
# also save the metadata table (collections) to CSV
collection_manga.to_csv("cleaned_manga_collections_with_combined_text.csv", index=False)
# this is optional
np.save("embeddings_manga.npy", embeddings_manga)

Vectors in index: 128507
